# Driver Drowsiness Detection
### A Deep Learning project using PyTorch, OpenCV, scikit-learn and YOLOv8

**Goal** — Monitor a driver's behaviour in real time and warn them (or a transport
company / passengers) before fatigue causes an accident.

**What the system does**

1. Detects the driver's face in a frame (OpenCV Haar cascade + YOLOv8 person check).
   **Detection only ever runs when a face is present in the current frame.**
2. Crops the **eyes** and **mouth** regions.
3. Runs two small **CNN classifiers** (PyTorch):
   - `EyeCNN` — *open* vs *closed* eye, trained on the public **MRL Eye Dataset**
     (84,898 real eye images) and fine-tuned on the user's own webcam crops
   - `MouthCNN` — *yawn* vs *no yawn*
4. Yawn is only reported when the mouth **actually looks open** (mouth-openness
   contrast gate on top of the CNN), so a closed mouth is never labelled yawning.
5. Decides an alert level from frame-to-frame evidence:
   - **SAFE**        — eyes open, looking at the road
   - **DROWSY**      — both eyes closed for several consecutive frames (PERCLOS-style)
   - **YAWNING**     — mouth open wide for several consecutive frames

**How to run it**
- **Double-click `Drowsiness_Detector.bat`** -> a launcher window opens with three
  buttons: *Live Webcam*, *Upload Video*, *Upload Image* (built with Tkinter).
- Or run the notebook below, which supports the same three input modes.

**Libraries used** (workshop set only): `numpy`, `pandas`, `opencv-python`, `matplotlib`,
`scikit-learn`, `torch`/`torchvision`, and `ultralytics` (YOLO).

---
## 0. Setup

If any package is missing:

```python
# !pip install numpy pandas opencv-python matplotlib scikit-learn torch torchvision ultralytics
```


In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch
import torchvision

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, roc_curve, auc)

from detector import DrowsinessDetector, detect_image, detect_video, detect_webcam

print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("opencv", cv2.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")
print("numpy", np.__version__, "| pandas", pd.__version__)

---
## 1. Training dataset (generated synthetically with OpenCV)

We need labelled images of *open / closed eyes* and *yawning / normal mouths*.
Because downloading a large labelled driver dataset requires authentication, we
**generate a synthetic dataset** with OpenCV that mimics the *geometric crops*
the detector actually feeds to the network (skin + eyebrow + eye on top).

> The detector's region extraction is the *same* code used here, so the CNN is
> trained on images that look like the crops it will see at inference time.
>
> *(Optional)* real crops of *your* face can be added in Section 6 for even
> better accuracy on your machine.

In [ ]:
import generate_data

REGEN_DATA = False          # set True to regenerate the synthetic dataset from scratch
if REGEN_DATA:
    generate_data.make_dataset()

# show the resulting dataset as a table
rows = []
for folder in ["data/eyes/open", "data/eyes/closed", "data/mouth/yawn", "data/mouth/no_yawn"]:
    n = len([f for f in os.listdir(folder) if f.endswith(".png")])
    rows.append({"class": folder.replace("data/", ""), "images": n})
pd.DataFrame(rows)

#### Visual check of the generated data

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 9))
classes = {"eye_open": "data/eyes/open", "eye_closed": "data/eyes/closed",
           "mouth_yawn": "data/mouth/yawn", "mouth_no_yawn": "data/mouth/no_yawn"}
for r, (name, folder) in enumerate(classes.items()):
    files = sorted(os.listdir(folder))[:4]
    for c, f in enumerate(files):
        img = cv2.cvtColor(cv2.imread(os.path.join(folder, f)), cv2.COLOR_BGR2RGB)
        axes[r, c].imshow(img); axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_title(name, fontsize=11)
plt.tight_layout(); plt.show()

---
## 2. Model architecture

Both classifiers use the same compact CNN: three convolutional blocks
(conv -> batchnorm -> ReLU -> max-pool) followed by dropout and two dense layers.
It has only **~1.3 M parameters**, so it trains in minutes on a CPU and runs at
>30 FPS on a laptop.

In [ ]:
from train import SmallCNN

model = SmallCNN(num_classes=2)
n_params = sum(p.numel() for p in model.parameters())
print(f"EyeCNN/MouthCNN parameters: {n_params:,}")

# print one forward pass shape to visualise the pipeline
x = torch.randn(1, 3, 48, 48)
with torch.no_grad():
    print("input :", tuple(x.shape))
    print("output:", tuple(model(x).shape))   # 2 logits: class scores

---
## 3. Training the CNNs

Each network is trained for 15 epochs with Adam (lr 1e-3, step-decay), using
train-time augmentation (rotation, translation, brightness/contrast jitter).
A held-out 20% split is used for validation.

> Running from scratch takes **~8-10 minutes on CPU**. Pre-trained weights are
> already saved in `models/`, so the notebook runs instantly. Set
> `TRAIN_MODELS = True` below to retrain.

In [ ]:
import train as T

TRAIN_MODELS = False        # set True to retrain (~10 min on CPU)

if TRAIN_MODELS:
    eye_report = T.fit("eye",   "models/eye_cnn.pth",   "models/eye_history.png")
    mouth_report = T.fit("mouth", "models/mouth_cnn.pth", "models/mouth_history.png")
    from IPython.display import Image, display
    display(Image("models/eye_history.png"))
    display(Image("models/mouth_history.png"))
else:
    print("Using pre-trained weights in models/  (set TRAIN_MODELS = True to retrain).")
    eye_report = None

#### Evaluation on the validation set (scikit-learn metrics)

In [ ]:
from torch.utils.data import DataLoader

def evaluate_cnn(kind, weights_path, names):
    train_df, val_df, _ = T.build_dfs(kind)
    val_ds = T.ImageFolderDS(val_df, transform=T.VAL_TFM)
    loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)
    net = SmallCNN(2)
    net.load_state_dict(torch.load(weights_path, map_location="cpu"))
    net.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = net(xb)
            preds.extend(out.argmax(1).numpy())
            labels.extend(yb.numpy())
    return names, labels, preds

names_e, labels_e, preds_e = evaluate_cnn("eye", "models/eye_cnn.pth", ["open", "closed"])
names_m, labels_m, preds_m = evaluate_cnn("mouth", "models/mouth_cnn.pth", ["yawn", "no_yawn"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (names, y_true, y_pred, title) in zip(axes, [
        (names_e, labels_e, preds_e, "Eye CNN"),
        (names_m, labels_m, preds_m, "Mouth CNN")]):
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=names).plot(ax=ax, cmap="Blues")
    ax.set_title(f"{title} - confusion matrix")
plt.tight_layout(); plt.show()

for names, y_true, y_pred, title in [(names_e, labels_e, preds_e, "Eye"),
                                     (names_m, labels_m, preds_m, "Mouth")]:
    print(f"{title:6s} acc={accuracy_score(y_true, y_pred):.3f} "
          f"prec={precision_score(y_true, y_pred, zero_division=0):.3f} "
          f"rec={recall_score(y_true, y_pred, zero_division=0):.3f} "
          f"f1={f1_score(y_true, y_pred, zero_division=0):.3f}")

> **Note on the 100% score** — the synthetic classes are deliberately
> well-separated (open eyes contain a dark iris, closed eyes are clean skin), so
> the CNN converges to perfect validation accuracy on them. For the *real world*
> the eye CNN is additionally trained on the public MRL Eye Dataset and then
> fine-tuned on the user's own webcam crops (Sections 3.5 and 6).

---
## 3.5 Training the eye CNN on a real public dataset (MRL Eye Dataset)

The synthetic eye set generalises poorly to real webcam light (a dark iris is
invisible in some light). To fix this, the eye CNN is first trained on the
**MRL Eye Dataset** — 84,898 real eye images (63,173 closed / 21,725 open),
each named like `s0001_00001_<eye-state>_...png` where `0` = closed and `1` = open.
We balance 18,000 images per class and train the same `SmallCNN` for 15 epochs
with brightness/contrast augmentation, then fine-tune on the user's own crops.

The trained weights are already in `models/eye_cnn_mrl.pth`. Set `TRAIN_MRL = True`
to retrain from the extracted dataset in `data/mrl/` (needs a CUDA GPU or ~2 h CPU;
the cached pipeline in `src/train_mrl.py` keeps every epoch under a minute on GPU).

In [ ]:
import train_mrl

TRAIN_MRL = False       # set True to retrain on MRL (GPU recommended)
if TRAIN_MRL:
    train_mrl.main()

# quick sanity evaluation of the saved MRL model on a subset
import numpy as np
from train_mrl import get_cached, CACHE_VAL, VAL_TFM
from train import SmallCNN, DEVICE
x_val, y_val = get_cached(None, CACHE_VAL)
net = SmallCNN(2); net.load_state_dict(torch.load("models/eye_cnn_mrl.pth", map_location="cpu"))
net.to(DEVICE).eval()
with torch.no_grad():
    out = net(VAL_TFM(x_val).to(DEVICE))
preds = out.argmax(1).cpu().numpy(); labels = y_val.numpy()
print(f"MRL eye CNN | val accuracy = {accuracy_score(labels, preds):.4f} "
      f"(open-vs-closed, {len(labels):,} real eyes)")

---
## 4. Detection engine

The `DrowsinessDetector` combines everything:

| Component | Library | Job |
|---|---|---|
| Haar cascade (`haarcascade_frontalface_default`) | OpenCV | find the face box |
| Geometric regions | OpenCV | crop eyes + mouth inside the face |
| `EyeCNN` (MRL + fine-tuned) + `MouthCNN` | PyTorch | classify each crop |
| Mouth-openness contrast gate | OpenCV/numpy | yawn is only reported when the mouth looks open |
| `yolov8n` person detector | YOLO (ultralytics) | verify a driver is in the frame |
| Frame-to-frame counters | Python | turn per-frame predictions into alerts |

Alert logic: `both eyes closed >= 3 frames -> DROWSY` (beeps once on Windows live),
`yawn (mouth open) >= 2 frames -> YAWNING`.
Eye/mouth classification only runs when a face is detected in the current frame.

In [ ]:
detector = DrowsinessDetector(use_yolo=True, yolo_interval=3)
print("face cascade :", "loaded" if not detector.face_cascade.empty() else "MISSING")
print("EyeCNN       :", "loaded" if detector.eye_net is not None else "MISSING")
print("MouthCNN     :", "loaded" if detector.mouth_net is not None else "MISSING")
print("YOLOv8       :", "loaded" if detector.yolo is not None else "off")
print("fusion weight:", detector.eye_cnn_weight, "| real model:", detector.using_real_model)

---
## 5. Detection — all three input modes

### 5.1 Upload an **image**

Run the next cell to show an upload button, then pick an image of a person and run
the following cell. If nothing is uploaded, the bundled sample image is used.

In [ ]:
from ipywidgets import FileUpload
upload_img = FileUpload(accept="image/*", multiple=False, description="Upload image")
display(upload_img)

In [ ]:
def run_uploaded_image(upload_widget, fallback="samples/sample_face.jpg"):
    if upload_widget.value:
        name = list(upload_widget.value.keys())[0]
        content = list(upload_widget.value.values())[0]["content"]
        with open("outputs/uploaded_image.jpg", "wb") as f:
            f.write(content)
        path = "outputs/uploaded_image.jpg"
        print(f"processing uploaded image: {name}")
    else:
        path = fallback
        print(f"no upload detected - using sample image: {fallback}")

    annotated, status = detect_image(detector, path, "outputs/annotated_image.jpg")
    img = cv2.cvtColor(cv2.imread("outputs/annotated_image.jpg"), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(7, 5))
    plt.imshow(img); plt.axis("off"); plt.title("Detection result"); plt.show()
    print("status:", status)

run_uploaded_image(upload_img)

### 5.2 Upload a **video**

Same idea for video files (mp4 / avi / mov). The annotated video is written to
`outputs/annotated_video.mp4` and shown inline.

In [ ]:
upload_vid = FileUpload(accept="video/*", multiple=False, description="Upload video")
display(upload_vid)

In [ ]:
from IPython.display import Video, display as ipydisplay

def run_uploaded_video(upload_widget, fallback="samples/sample_driver_video.mp4"):
    if upload_widget.value:
        name = list(upload_widget.value.keys())[0]
        content = list(upload_widget.value.values())[0]["content"]
        with open("outputs/uploaded_video.mp4", "wb") as f:
            f.write(content)
        path = "outputs/uploaded_video.mp4"
        print(f"processing uploaded video: {name}")
    else:
        path = fallback
        print(f"no upload detected - using sample video: {fallback}")

    stats = detect_video(detector, path, "outputs/annotated_video.mp4")
    print("per-frame stats:", stats)
    print("saved -> outputs/annotated_video.mp4")
    return stats

stats = run_uploaded_video(upload_vid)
ipydisplay(Video("outputs/annotated_video.mp4", embed=True, width=640))

### 5.3 **Live webcam** detection

Run the cell below, then look at the camera. An OpenCV window opens showing the
live annotated feed — **press `q` inside that window to stop**.

Try it:
- **close your eyes** for a couple of seconds  -> red **DROWSY** + beep
- **open your mouth wide** (yawn)              -> orange **YAWNING**

In [ ]:
# the webcam window opens on your screen; press 'q' in it to stop
detect_webcam(detector, camera=0)

---
## 6. (Optional) Calibrate on YOUR face for best accuracy

The pre-trained CNNs were trained on synthetic crops. For a personal, very
reliable demo you can capture **real** crops of your own eyes and mouth from the
webcam in ~30 seconds, then fine-tune both CNNs on them. The detector
automatically prefers the fine-tuned weights (`*_real.pth`).

**Step 1 — guided capture.** Follow the on-screen prompts
(eyes open -> eyes closed -> yawn -> mouth relaxed).

In [ ]:
import collect_calibration
collect_calibration.main()   # ~30 s: follow the prompts in the webcam window

**Step 2 — fine-tune** on the real crops. The eye CNN starts from the MRL
weights (when `models/eye_cnn_mrl.pth` exists), the mouth CNN from its synthetic
weights; synthetic data is mixed in for regularisation.

In [ ]:
import finetune
finetune.main()   # retrains both *_real.pth models and prints held-out real accuracy

# recreate the detector - it now loads the real fine-tuned models automatically
detector = DrowsinessDetector(use_yolo=True, yolo_interval=3)
print("now using real-data model:", detector.using_real_model)

---
## 7. Results on real data (this machine)

A 20-second webcam video of the driver (eyes open -> both eyes closed -> yawn ->
relaxed) was recorded and processed offline with the final detector:

| Metric | Value |
|---|---|
| Frames processed | 585 (~20 s @ ~30 fps) |
| DROWSY frames | 345 (eyes kept closed after the drowsiness segment) |
| YAWNING frames | 65 (mouth-open yawn segment correctly detected) |

The live webcam test also passed: SAFE with eyes open, DROWSY with both eyes
closed, YAWNING when yawning — the misclassification that originally sent the
CNN to zero probability on real open eyes is fixed by the MRL-trained eye model.
On the user's held-out calibration crops both fine-tuned CNNs reach 100% accuracy.

---
## Conclusion

This project delivers a complete driver drowsiness detection
pipeline using only the workshop library set:

- **Dataset**: a synthetic dataset of eye/mouth crops generated with OpenCV
  (4800 images, 4 classes) **plus** the public MRL Eye Dataset (84,898 real eye
  images) used to train the eye CNN on real-world lighting.
- **Deep learning**: two small CNNs (PyTorch) classify eye open/closed and
  yawn/no-yawn. Eye CNN: **99.8%** on 5,400 held-out MRL eyes; **100%** on the
  user's held-out real calibration crops (both CNNs).
- **Detection engine**: Haar face detection + geometric crops + CNN + YOLOv8
  driver-presence check + temporal alert logic (PERCLOS-style). Classification
  only runs when a face is present, and yawn is only reported when the mouth
  looks open (contrast gate).
- **Three input modes**: uploaded image, uploaded video and live webcam — from
  both the notebook and a one-click **Tkinter GUI** (`Drowsiness_Detector.bat`).

**Stakeholders**: drivers get an immediate warning, transport companies get
telematics on risky behaviour, and passengers get a safer ride.

**Limitations & future work**
- Real-time performance depends on webcam auto-exposure; very dark scenes are
  still hard (mitigated by training with brightness/contrast augmentation).
- Gaze direction (looking away = distracted) could be added with a landmark model.
- A mobile/embedded deployment could run the same ONNX-exported models.

*Project files:* `src/` (data generation, MRL training, CNN training, fine-tuning,
detection, calibration, GUI), `models/` (weights), `data/` (synthetic + calibration +
MRL dataset), `demo/` (recorded demo + annotated result), `outputs/` (annotated results).